In [61]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import SelectKBest, f_regression
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [62]:
"""Load data and print initial statistics"""
print("="*80)
print("SECTION 1: DATA LOADING & EXPLORATION")
print("="*80)
    
df = pd.read_parquet('D:/Study/data_science/underpriced-listing-predictor/data/03.cleaned/multi_appliances_cleaned.parquet')
    
print(f"\nDataset shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    
# Price distribution analysis
price = df['price']
print(f"\n--- TARGET VARIABLE (Price) Statistics ---")
print(f"Mean: ₹{price.mean():.2f}")
print(f"Median: ₹{price.median():.2f}")
print(f"Std Dev: ₹{price.std():.2f}")
print(f"Range: ₹{price.min():.0f} - ₹{price.max():.0f}")
print(f"Skewness: {stats.skew(price):.4f} (Highly right-skewed!)")
print(f"Kurtosis: {stats.kurtosis(price):.4f} (Heavy tails with outliers!)")
    
# Category distribution
print(f"\n--- Category Distribution ---")
print(df['category'].value_counts())


SECTION 1: DATA LOADING & EXPLORATION

Dataset shape: (2900, 54)
Rows: 2900, Columns: 54

--- TARGET VARIABLE (Price) Statistics ---
Mean: ₹33386.40
Median: ₹30990.00
Std Dev: ₹22406.79
Range: ₹3990 - ₹397990
Skewness: 5.3681 (Highly right-skewed!)
Kurtosis: 62.9704 (Heavy tails with outliers!)

--- Category Distribution ---
category
AC                1009
Refrigerator      1008
WashingMachine     883
Name: count, dtype: int64


**Target Column Transformation**

In [63]:
"""Apply log transformation to address extreme skewness"""
print("\n" + "="*80)
print("SECTION 2: TARGET VARIABLE TRANSFORMATION")
print("="*80)
    
# Log transformation
df['log_price'] = np.log1p(df['price'])
    
# Verify improvement
original_skew = stats.skew(df['price'])
transformed_skew = stats.skew(df['log_price'])
original_kurtosis = stats.kurtosis(df['price'])
transformed_kurtosis = stats.kurtosis(df['log_price'])
    
print(f"\n--- Transformation Results ---")
print(f"Original Price Skewness: {original_skew:.4f}")
print(f"Log Price Skewness: {transformed_skew:.4f}")
print(f"Improvement: {((original_skew - transformed_skew) / original_skew * 100):.1f}%")
    
print(f"\nOriginal Kurtosis: {original_kurtosis:.4f}")
print(f"Log Price Kurtosis: {transformed_kurtosis:.4f}")
print(f"Improvement: {((original_kurtosis - transformed_kurtosis) / original_kurtosis * 100):.1f}%")


SECTION 2: TARGET VARIABLE TRANSFORMATION

--- Transformation Results ---
Original Price Skewness: 5.3681
Log Price Skewness: -0.0346
Improvement: 100.6%

Original Kurtosis: 62.9704
Log Price Kurtosis: 0.6449
Improvement: 99.0%


**Missing Data Imputation**

In [64]:
"""
    Category-aware missing data handling:
    - Category-specific features (ac_*, ref_*, wm_*) → fill with 0 for non-matching categories
    - Other features (star_rating) → impute with category median
    - Low-information features → drop
"""
print("\n" + "="*80)
print("SECTION 3: MISSING DATA HANDLING")
print("="*80)
    
# Identify category-specific features
ac_features = [col for col in df.columns if col.startswith('ac_')]
ref_features = [col for col in df.columns if col.startswith('ref_')]
wm_features = [col for col in df.columns if col.startswith('wm_')]
    
print(f"\nIdentified feature categories:")
print(f"  AC-specific features: {len(ac_features)}")
print(f"  Refrigerator-specific features: {len(ref_features)}")
print(f"  WashingMachine-specific features: {len(wm_features)}")
    
# ===== STEP 1: Category-aware zero imputation =====
print(f"\n--- Step 1: Category-Aware Zero Imputation ---")
    
# For ACs, fill ref_* and wm_* with 0
df.loc[df['category'] == 'AC', ref_features] = df.loc[df['category'] == 'AC', ref_features].fillna(0)
df.loc[df['category'] == 'AC', wm_features] = df.loc[df['category'] == 'AC', wm_features].fillna(0)
    
# For Refrigerators, fill ac_* and wm_* with 0
df.loc[df['category'] == 'Refrigerator', ac_features] = df.loc[df['category'] == 'Refrigerator', ac_features].fillna(0)
df.loc[df['category'] == 'Refrigerator', wm_features] = df.loc[df['category'] == 'Refrigerator', wm_features].fillna(0)
    
# For WashingMachines, fill ac_* and ref_* with 0
df.loc[df['category'] == 'WashingMachine', ac_features] = df.loc[df['category'] == 'WashingMachine', ac_features].fillna(0)
df.loc[df['category'] == 'WashingMachine', ref_features] = df.loc[df['category'] == 'WashingMachine', ref_features].fillna(0)
    
print("✓ Category-specific features filled with 0 for non-matching categories")
    
# ===== STEP 2: Impute other numeric columns with category median =====
print(f"\n--- Step 2: Median Imputation by Category ---")
    
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('price')
numeric_cols.remove('log_price')
    
for col in numeric_cols:
    if df[col].isna().sum() > 0:
        df[col] = df.groupby('category')[col].transform(lambda x: x.fillna(x.median()))
    
print("✓ Other numeric columns imputed with category median")
    
# ===== STEP 3: Drop low-information features =====
print(f"\n--- Step 3: Drop Low-Information Features ---")
    
# Features with >70% missing that won't be useful
cols_to_drop = ['model_year', 'is_recent_model', 'appliance_age']
df = df.drop(columns=cols_to_drop, errors='ignore')
    
print(f"✓ Dropped columns: {cols_to_drop}")

# fill star rating of Washing Machines as 0
df['star_rating'] = df['star_rating'].fillna(0)
    
# Final missing data check
remaining_missing = df.isna().sum().sum()
print(f"\nRemaining missing values: {remaining_missing}")


SECTION 3: MISSING DATA HANDLING

Identified feature categories:
  AC-specific features: 10
  Refrigerator-specific features: 14
  WashingMachine-specific features: 13

--- Step 1: Category-Aware Zero Imputation ---
✓ Category-specific features filled with 0 for non-matching categories

--- Step 2: Median Imputation by Category ---
✓ Other numeric columns imputed with category median

--- Step 3: Drop Low-Information Features ---
✓ Dropped columns: ['model_year', 'is_recent_model', 'appliance_age']

Remaining missing values: 0


In [65]:
df.isna().sum()

price                       0
rating                      0
category                    0
n_features                  0
brand_name                  0
capacity_value              0
capacity_unit               0
star_rating                 0
has_star_rating             0
has_inverter                0
has_wifi                    0
has_voice_control           0
has_app_control             0
smart_connectivity_score    0
ac_split                    0
ac_window                   0
ac_pm25_filter              0
ac_hepa_filter              0
ac_auto_clean               0
ac_hot_and_cold             0
ac_copper_condenser         0
ac_Dehumidification         0
ac_Turbo Mode               0
ac_Self Diagnosis           0
ref_single_door             0
ref_multi_door              0
ref_chest_freezer           0
ref_side_door               0
ref_french_door             0
ref_double_door             0
ref_triple_door             0
ref_frost_free              0
ref_convertible             0
ref_door_a

### ***Feature Engineering***

In [66]:
# 1. Polynomial features for top correlates
df['capacity_sq'] = df['capacity_value'] ** 2
df['n_features_sq'] = df['n_features'] ** 2
df['rating_sq'] = df['rating'] ** 2
print(" Polynomial features: capacity_sq, n_features_sq, rating_sq")

 Polynomial features: capacity_sq, n_features_sq, rating_sq


In [67]:
# 2. Interaction terms
df['capacity_n_features'] = df['capacity_value'] * df['n_features']
df['capacity_rating'] = df['capacity_value'] * df['rating']
df['rating_n_features'] = df['rating'] * df['n_features']
print("Interaction terms: capacity_n_features, capacity_rating, rating_n_features")

Interaction terms: capacity_n_features, capacity_rating, rating_n_features


In [68]:
# 3. Smart connectivity features
df['smart_intensity'] = (df['has_wifi'].fillna(0) + 
                        df['has_app_control'].fillna(0) + 
                        df['has_voice_control'].fillna(0) +
                        df['smart_connectivity_score'].fillna(0))
print("Smart connectivity features: smart_intensity")

Smart connectivity features: smart_intensity


In [69]:
# 4. Category-specific premium features
df['ac_premium_features'] = (df['ac_hot_and_cold'].fillna(0) + 
                            df['ac_auto_clean'].fillna(0) + 
                            df['ac_copper_condenser'].fillna(0))
    
df['ref_door_complexity'] = (df['ref_french_door'].fillna(0) + 
                            df['ref_side_door'].fillna(0) + 
                            df['ref_triple_door'].fillna(0))
    
df['wm_tech_level'] =  (df['wm_front_load'].fillna(0) * 2 +  # Front-load is premium
                        df['wm_inbuilt_heater'].fillna(0) +
                        df['wm_quick_wash'].fillna(0))
print("Category-specific features: ac_premium_features, ref_door_complexity, wm_tech_level")

Category-specific features: ac_premium_features, ref_door_complexity, wm_tech_level


In [70]:
# 5. Relative features (compared to category average)
df['features_above_avg'] = df['n_features'] - df.groupby('category')['n_features'].transform('mean')
df['rating_above_avg'] = df['rating'] - df.groupby('category')['rating'].transform('mean')
df['capacity_above_avg'] = df['capacity_value'] - df.groupby('category')['capacity_value'].transform('mean')
print("Relative features: features_above_avg, rating_above_avg, capacity_above_avg")

Relative features: features_above_avg, rating_above_avg, capacity_above_avg


In [71]:
# 6. Brand frequency
df['brand_frequency'] = df['brand_name'].map(df['brand_name'].value_counts())


### ***Outlier Removal***

In [76]:
# 1. Drop the top 1% outlier prices per category
# We group by category so a 99th percentile AC is calculated separately from a 99th percentile Refrigerator.
q99_thresholds = df.groupby('category')['price'].transform(lambda x: x.quantile(0.99))
data_clean = df[df['price'] <= q99_thresholds].copy()

print(f"Original rows: {len(df)}")
print(f"Cleaned rows:  {len(data_clean)}")
print(f"Dropped {len(df) - len(data_clean)} extreme outliers.\\n")

# 2. Split 'capacity_value' into orthogonal features based on their unit
data_clean['capacity_ac_tons'] = np.where(
    data_clean['capacity_unit'].str.contains('Ton', case=False, na=False), 
    data_clean['capacity_value'], 
    0.0
)

data_clean['capacity_ref_liters'] = np.where(
    data_clean['capacity_unit'].str.contains('L', case=False, na=False), 
    data_clean['capacity_value'], 
    0.0
)

data_clean['capacity_wm_kg'] = np.where(
    data_clean['capacity_unit'].str.contains('kg', case=False, na=False), 
    data_clean['capacity_value'], 
    0.0
)

# 3. Drop the old conflicting columns to prevent data leakage/confusion
data_clean = data_clean.drop(columns=['capacity_value', 'capacity_unit'])


Original rows: 2900
Cleaned rows:  2870
Dropped 30 extreme outliers.\n


In [77]:
print(f"\nTotal features created: {len(data_clean.columns) - 2}")  # -2 for price, log_price


Total features created: 65


In [78]:
data_clean.info()

<class 'pandas.DataFrame'>
Index: 2870 entries, 0 to 2924
Data columns (total 67 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   price                     2870 non-null   int64  
 1   rating                    2870 non-null   float64
 2   category                  2870 non-null   str    
 3   n_features                2870 non-null   int64  
 4   brand_name                2870 non-null   str    
 5   star_rating               2870 non-null   float64
 6   has_star_rating           2870 non-null   int64  
 7   has_inverter              2870 non-null   int64  
 8   has_wifi                  2870 non-null   int64  
 9   has_voice_control         2870 non-null   int64  
 10  has_app_control           2870 non-null   int64  
 11  smart_connectivity_score  2870 non-null   int64  
 12  ac_split                  2870 non-null   Int8   
 13  ac_window                 2870 non-null   Int8   
 14  ac_pm25_filter          

In [79]:
data_clean.to_parquet('D:/Study/data_science/underpriced-listing-predictor/data/03.cleaned/multi_appliances_cleaned_engineered.parquet')
data_clean.to_csv('D:/Study/data_science/underpriced-listing-predictor/data/03.cleaned/multi_appliances_cleaned_engineered.csv')